# Outlier — Week 1: Data Validation

**Goal:** confirm the data foundation before any modeling — that our scene segmentation lines up with TRIPOD's gold turning-point labels, that the labels load, and that the evaluation metric behaves.

This notebook uses the `outlier` package in `src/`. Path B stack: sentence-transformers (later) instead of USE; **no old TRIPOD/USE environment required.**

In [5]:
import sys
from pathlib import Path
# locate the repo root (the folder containing src/outlier) whether run from repo root or notebooks/
here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / 'src' / 'outlier').exists()), here)
sys.path.insert(0, str(root / 'src'))

try:
    import pandas as pd
except ModuleNotFoundError:
    import sys
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas"])
    import pandas as pd
from outlier import data, metrics, config
print('repo root:', root)
print('turning points:', config.TP_NAMES)

repo root: c:\Users\porte\OneDrive - Neumont College of Computer Science\Documents\.AllRelevantProjects\aml-script-analyzer
turning points: ['Opportunity', 'Change of Plans', 'Point of No Return', 'Major Setback', 'Climax']


## 1. Gold labels & segmentation alignment

The 15 test movies have **scene-level** gold turning points. Our segmenter must produce scene indices that match — i.e. every gold TP index must be < the movie's scene count.

In [6]:
gold = data.load_gold_labels()
rows = []
for mv, tps in gold.items():
    scenes = data.get_scenes(mv)
    max_idx = max(i for tp in tps for i in tp)
    rows.append({
        'movie': mv,
        'scenes': len(scenes),
        'tp_scenes': [tp[0] for tp in tps],
        'max_gold_idx': max_idx,
        'aligned': max_idx < len(scenes),
    })
df = pd.DataFrame(rows)
print('all aligned:', bool(df['aligned'].all()))
df

all aligned: True


,movie,scenes,tp_scenes,max_gold_idx,aligned
0,The Back-up Plan,132,"[9, 40, 82, 106, 131]",131,True
1,The Shining (film),230,"[6, 38, 76, 177, 223]",229,True
2,Juno (film),116,"[3, 31, 39, 86, 107]",107,True
3,Soldier (1998 American film),230,"[35, 51, 109, 210, 223]",223,True
4,Panic Room,167,"[17, 56, 135, 148, 159]",160,True
5,Arbitrage (film),110,"[35, 57, 67, 105, 109]",109,True
6,The Breakfast Club,42,"[5, 20, 31, 31, 34]",39,True
7,Slumdog Millionaire,196,"[32, 108, 139, 150, 188]",191,True
8,Total Recall (1990 film),163,"[16, 51, 72, 112, 145]",154,True
9,Unforgiven,106,"[6, 53, 88, 95, 98]",98,True


## 2. Turning points on a real film

Sanity check: do the labelled scenes actually look like the beats they claim to be? Here are Die Hard's five turning points (first line of each scene).

In [7]:
mv = 'Die Hard'
scenes = data.get_scenes(mv)
for name, tp in zip(config.TP_NAMES, gold[mv]):
    idx = tp[0]
    first = scenes[idx].strip().splitlines()[0][:70]
    print(f'{name:20} scene {idx:>3}: {first}')

Opportunity          scene  11: 21      INT. ELLIS' OFFICE - NIGHT                             21 
Change of Plans      scene  26: 52      INT. 30th FLOOR (HOSTAGE FLOOR) - SAME                 52
Point of No Return   scene  99: 227     INT. 33RD FLOOR - ON MCCLANE                         227
Major Setback        scene 114: 346     INT. VAULT ROOM                                       346
Climax               scene 116: 393     EXT. BUILDING - LONG SHOT                             393


## 3. Silver training labels

Projected (distant-supervision) TP scene indices for the training movies, from SUMMER. Same shape as gold: `{movie: [tp1..tp5]}`. (If this cell errors with *truncated*, the Git-LFS file is still syncing — re-run once it's fully downloaded.)

In [8]:
try:
    silver = data.load_silver_labels()
    on_disk = set(data.list_movies())
    have_script = sum(1 for m in silver if m in on_disk)
    print(f'silver movies: {len(silver)} | with a screenplay on disk: {have_script}')
    ex = next(iter(silver))
    print('example:', ex, '->', silver[ex])
except Exception as e:
    print('silver not loadable yet (LFS still syncing?):', e)

silver movies: 84 | with a screenplay on disk: 81
example: 10 Things I Hate About You -> [[20, 21, 22], [29, 30, 31], [43, 44, 45], [66, 67, 68], [84, 85, 86]]


## 4. Metric sanity check

`TA` = exact agreement, `PA` = within a small window, `D` = normalized distance (lower is better). A perfect prediction should score TA=1.0 / D=0.0; a random one should not.

In [9]:
g = gold['Die Hard']
n = len(data.get_scenes('Die Hard'))
perfect = [tp[0] for tp in g]
print('perfect :', metrics.evaluate(perfect, g, n))
print('all-zero:', metrics.evaluate([0, 0, 0, 0, 0], g, n))

perfect : {'TA': 1.0, 'PA': 1.0, 'D': 0.0}
all-zero: {'TA': 0.0, 'PA': 0.0, 'D': 0.6203389830508474}


## Next (Week 3)

1. Encode each scene with **sentence-transformers**.
2. Train a small **scene-sequence model** on the silver labels; evaluate on the 15 gold movies with the metrics above.
3. Track runs in **MLflow**.

The scene-turn and escalation (secondary principles) build on the same scenes and need no TRIPOD labels.